# Sparse Autoencoders: Variants and Implementations

**Survey Reference:** Section 7.1 - Sparse Autoencoders: The Foundation

## Overview

Feature extraction identifies distributed features across neurons that correspond to coherent semantic units. Sparse autoencoders (SAEs) address polysemanticity (neurons encoding multiple features) by decomposing activations into sparse, interpretable features using an overcomplete basis ($m >> d$).

**Core SAE formulation:**
- **Encode:** $f = \sigma(W_e x + b_e)$ where $f \in \mathbb{R}^m$ is sparse
- **Decode:** $\hat{x} = W_d f + b_d$

This notebook implements and compares five SAE variants:

1. **Basic SAE (ReLU)** - Standard L1-penalized autoencoder
2. **Gated SAE** - Separates magnitude and gating for better feature learning
3. **JumpReLU SAE** - Uses straight-through estimator for discrete features
4. **Top-K SAE** - Activates exactly K features per example
5. **Batch Top-K SAE** - Distributes K activations across the batch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Basic SAE Architecture (ReLU)

The standard SAE has:
- **Expansion factor:** m = expansion * d
- **ReLU activation:** Creates sparsity in the feature space
- **L1 penalty:** Encourages sparse feature activations
- **Tied or untied weights:** $W_d = W_e^T$ or learned separately

### Architecture:
$$f = \text{ReLU}(W_e x + b_e)$$
$$\hat{x} = W_d f + b_d$$

Loss: $\mathcal{L} = \|x - \hat{x}\|_2^2 + \lambda \|f\|_1$

In [ ]:
class SparseAutoencoder(nn.Module):
    """
    A basic Sparse Autoencoder for mechanistic interpretability.
    
    Architecture:
        input (d) -> encoder (d -> m) -> ReLU -> decoder (m -> d)
    """
    
    def __init__(self, d_model: int, n_features: int, tied_weights: bool = False):
        super().__init__()
        self.d_model = d_model
        self.n_features = n_features
        self.tied_weights = tied_weights
        
        # Encoder
        self.W_enc = nn.Parameter(torch.randn(d_model, n_features) / np.sqrt(d_model))
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        
        # Decoder
        if tied_weights:
            self.W_dec = None  # Will use W_enc.T
        else:
            self.W_dec = nn.Parameter(torch.randn(n_features, d_model) / np.sqrt(n_features))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
    
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode input to sparse feature activations."""
        # Center the input (optional but common)
        x_centered = x - self.b_dec
        # Linear projection + ReLU
        pre_acts = x_centered @ self.W_enc + self.b_enc
        return F.relu(pre_acts)
    
    def decode(self, f: torch.Tensor) -> torch.Tensor:
        """Decode feature activations back to input space."""
        if self.tied_weights:
            W_d = self.W_enc.T
        else:
            W_d = self.W_dec
        return f @ W_d + self.b_dec
    
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Full forward pass. Returns (reconstruction, feature_activations)."""
        f = self.encode(x)
        x_hat = self.decode(f)
        return x_hat, f
    
    @property
    def feature_directions(self) -> torch.Tensor:
        """Get normalized decoder directions for each feature."""
        if self.tied_weights:
            W = self.W_enc.T
        else:
            W = self.W_dec
        return W / W.norm(dim=1, keepdim=True)

## 2. Gated SAE

**Key Innovation:** Separates feature gating from magnitude estimation.

Traditional SAEs use ReLU which couples:
- Whether a feature is active (gating)
- How strong the activation is (magnitude)

**Gated SAE Architecture (from paper):**

$$\hat{f}(x) = \mathbb{1}[(W_{\text{gate}}(x - b_{\text{dec}}) + b_{\text{gate}}) > 0] \odot \text{ReLU}(W_{\text{mag}}(x - b_{\text{dec}}) + b_{\text{mag}})$$

Where:
- $\mathbb{1}[\cdot > 0]$ is the Heaviside step function (binary gating)
- $W_{\text{mag}}$ is tied to $W_{\text{gate}}$ via rescaling: $(W_{\text{mag}})_{ij} := \exp(r_{\text{mag}})_i \cdot (W_{\text{gate}})_{ij}$
- $\pi_{\text{gate}}(x)$ are the pre-activations used in the loss

**Loss Function**

$$\mathcal{L}_{\text{gated}} = \|\mathbf{x} - \hat{\mathbf{x}}\|_2^2 + \lambda \|\text{ReLU}(\pi_{\text{gate}})\|_1 + \|\mathbf{x} - \hat{\mathbf{x}}_{\text{frozen}}\|_2^2$$

The auxiliary loss $\|\mathbf{x} - \hat{\mathbf{x}}_{\text{frozen}}\|_2^2$ is crucial because the Heaviside step has zero gradients, so this provides gradient flow to the gating network through ReLU(π_gate) using a frozen copy of the decoder.

**Benefits:**
- Separates whether a feature is active (gating) from how strong (magnitude)
- Less shrinkage (L1 penalty only on gates, not magnitudes)
- Auxiliary loss enables training despite non-differentiable step function
- Weight tying reduces parameters

In [ ]:
class GatedSAE(nn.Module):
    """
    Reference: "Improving Dictionary Learning with Gated Sparse Autoencoders"
    
    L = ||x - x_hat||^2 + λ ||ReLU(pi_gate)||_1 + ||x - x_hat_frozen||^2
    """
    
    def __init__(self, d_model: int, n_features: int):
        super().__init__()
        self.d_model = d_model
        self.n_features = n_features
        
        # Gating network
        self.W_gate = nn.Parameter(torch.randn(d_model, n_features) / np.sqrt(d_model))
        self.b_gate = nn.Parameter(torch.zeros(n_features))
        
        # Magnitude network (tied to gating via rescaling parameter)
        self.r_mag = nn.Parameter(torch.zeros(n_features))  # Rescaling parameter
        self.b_mag = nn.Parameter(torch.zeros(n_features))
        
        # Decoder
        self.W_dec = nn.Parameter(torch.randn(n_features, d_model) / np.sqrt(n_features))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        
        # Frozen decoder copy for auxiliary loss (no gradients)
        # These are updated manually, not by optimizer
        self.register_buffer('W_dec_frozen', self.W_dec.detach().clone())
        self.register_buffer('b_dec_frozen', self.b_dec.detach().clone())
    
    def update_frozen_decoder(self):
        self.W_dec_frozen.copy_(self.W_dec.detach())
        self.b_dec_frozen.copy_(self.b_dec.detach())
    
    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Encode with separate gating and magnitude."""
        x_centered = x - self.b_dec
        
        # Gating pre-activations
        pi_gate = x_centered @ self.W_gate + self.b_gate
        
        # Magnitude network uses tied weights with rescaling
        # W_mag = exp(r_mag) * W_gate
        W_mag = torch.exp(self.r_mag).unsqueeze(0) * self.W_gate  # Broadcasting
        f_mag_pre = x_centered @ W_mag + self.b_mag
        
        # Gating: Heaviside step function (binary)
        gate = (pi_gate > 0).float()
        
        # Magnitude: ReLU
        mag = F.relu(f_mag_pre)
        
        # Combined features
        f = gate * mag
        
        return f, pi_gate
    
    def decode(self, f: torch.Tensor, use_frozen: bool = False) -> torch.Tensor:
        """
        Decode features back to input space.
        
        Args:
            f: Feature activations
            use_frozen: If True, use frozen decoder (for auxiliary loss)
        """
        if use_frozen:
            return f @ self.W_dec_frozen + self.b_dec_frozen
        else:
            return f @ self.W_dec + self.b_dec
    
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns (reconstruction, features, gate_pre_activations, aux_reconstruction).
        
        The aux_reconstruction uses frozen decoder on ReLU(pi_gate) for auxiliary loss.
        """
        f, pi_gate = self.encode(x)
        x_hat = self.decode(f)
        
        # Auxiliary reconstruction: decode ReLU of gate pre-activations with frozen decoder
        # This ensures gradients flow to W_gate and b_gate
        f_aux = F.relu(pi_gate)
        x_hat_frozen = self.decode(f_aux, use_frozen=True)
        
        return x_hat, f, pi_gate, x_hat_frozen

## 3. JumpReLU SAE

**Key Innovation:** Discrete feature activation with straight-through estimator.

Instead of soft sparsity (L1), JumpReLU uses a hard threshold:

$$f_i = \begin{cases} 
W_e^i x + b_e^i & \text{if } W_e^i x + b_e^i > \theta \\
0 & \text{otherwise}
\end{cases}$$

**Benefits:**
- True discrete sparsity (no shrinkage)
- Interpretable threshold $\theta$
- Uses straight-through estimator for gradients

**Forward:** Apply threshold
**Backward:** Pretend threshold doesn't exist (straight-through)

In [ ]:
class JumpReLU(torch.autograd.Function):
    """
    JumpReLU activation with straight-through estimator.
    
    Forward: f = x if x > threshold else 0
    Backward: grad_x = grad_f (as if threshold doesn't exist)
    """
    @staticmethod
    def forward(ctx, x, threshold):
        return x * (x > threshold).float()
    
    @staticmethod
    def backward(ctx, grad_output):
        # Straight-through: pass gradient as-is
        return grad_output, None


class JumpReLUSAE(nn.Module):
    """
    SAE using JumpReLU for discrete sparsity.
    
    Reference: "Jumping Through Local Minima" (2024)
    """
    
    def __init__(self, d_model: int, n_features: int, threshold: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_features = n_features
        self.threshold = threshold
        
        # Encoder
        self.W_enc = nn.Parameter(torch.randn(d_model, n_features) / np.sqrt(d_model))
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        
        # Decoder
        self.W_dec = nn.Parameter(torch.randn(n_features, d_model) / np.sqrt(n_features))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
    
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode with JumpReLU."""
        x_centered = x - self.b_dec
        pre_acts = x_centered @ self.W_enc + self.b_enc
        return JumpReLU.apply(pre_acts, self.threshold)
    
    def decode(self, f: torch.Tensor) -> torch.Tensor:
        """Decode features."""
        return f @ self.W_dec + self.b_dec
    
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Returns (reconstruction, features)."""
        f = self.encode(x)
        x_hat = self.decode(f)
        return x_hat, f

In [ ]:
class TopKSAE(nn.Module):
    """
    Top-K SAE: Activates exactly K features per example.
    
    Key idea: Replace ReLU/L1 with hard constraint - keep only top K activations.
    
    Benefits:
    - Exact sparsity control (no hyperparameter tuning)
    - No shrinkage bias
    - Efficient inference
    """
    
    def __init__(self, d_model: int, n_features: int, k: int):
        super().__init__()
        self.d_model = d_model
        self.n_features = n_features
        self.k = k
        
        # Encoder
        self.W_enc = nn.Parameter(torch.randn(d_model, n_features) / np.sqrt(d_model))
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        
        # Decoder
        self.W_dec = nn.Parameter(torch.randn(n_features, d_model) / np.sqrt(n_features))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
    
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode with top-K activation."""
        x_centered = x - self.b_dec
        pre_acts = F.relu(x_centered @ self.W_enc + self.b_enc)
        
        # Keep only top-K activations per example
        topk_vals, topk_indices = torch.topk(pre_acts, self.k, dim=-1)
        
        # Create sparse tensor
        f = torch.zeros_like(pre_acts)
        f.scatter_(-1, topk_indices, topk_vals)
        
        return f
    
    def decode(self, f: torch.Tensor) -> torch.Tensor:
        """Decode features."""
        return f @ self.W_dec + self.b_dec
    
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Returns (reconstruction, features)."""
        f = self.encode(x)
        x_hat = self.decode(f)
        return x_hat, f

In [ ]:
class BatchTopKSAE(nn.Module):
    """
    Batch Top-K SAE: Distributes K activations across the entire batch.
    
    Key difference from Top-K SAE:
    - Top-K: Each example has exactly K features
    - Batch Top-K: Entire batch has K*batch_size total activations
    
    Benefits:
    - More flexible sparsity patterns
    - Better utilization of features
    - Encourages feature specialization
    """
    
    def __init__(self, d_model: int, n_features: int, k: int):
        super().__init__()
        self.d_model = d_model
        self.n_features = n_features
        self.k = k  # Average k per example
        
        # Encoder
        self.W_enc = nn.Parameter(torch.randn(d_model, n_features) / np.sqrt(d_model))
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        
        # Decoder
        self.W_dec = nn.Parameter(torch.randn(n_features, d_model) / np.sqrt(n_features))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
    
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode with batch top-K activation."""
        batch_size = x.size(0)
        x_centered = x - self.b_dec
        pre_acts = F.relu(x_centered @ self.W_enc + self.b_enc)
        
        # Flatten to (batch_size * n_features,)
        pre_acts_flat = pre_acts.flatten()
        
        # Keep top K*batch_size activations across entire batch
        k_total = self.k * batch_size
        topk_vals, topk_indices = torch.topk(pre_acts_flat, k_total)
        
        # Create sparse tensor
        f_flat = torch.zeros_like(pre_acts_flat)
        f_flat[topk_indices] = topk_vals
        
        # Reshape back
        f = f_flat.reshape(batch_size, self.n_features)
        
        return f
    
    def decode(self, f: torch.Tensor) -> torch.Tensor:
        """Decode features."""
        return f @ self.W_dec + self.b_dec
    
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Returns (reconstruction, features)."""
        f = self.encode(x)
        x_hat = self.decode(f)
        return x_hat, f

## 4. Loss Functions

Different SAE variants use different loss functions.

In [ ]:
def compute_loss(x, x_hat, f, sae_type="basic", l1_coef=1e-3, pi=None, x_hat_aux=None):
    """
    Unified loss function for all SAE variants.
    
    Args:
        x: Input activations
        x_hat: Reconstructed activations
        f: Feature activations
        sae_type: "basic", "gated", "jumprelu", "topk", "batch_topk"
        l1_coef: L1 coefficient (for basic/gated)
        pi: Gate pre-activations (for gated SAE)
        x_hat_aux: Auxiliary reconstruction from frozen decoder (for gated SAE)
    
    Returns:
        total_loss, metrics dict
    """
    # Reconstruction loss (all variants)
    recon_loss = (x - x_hat).pow(2).mean()
    
    # Sparsity loss (variant-specific)
    if sae_type in ["basic", "jumprelu"]:
        # L1 penalty on features
        l1_loss = f.abs().mean()
        total_loss = recon_loss + l1_coef * l1_loss
        aux_loss = torch.tensor(0.0)
    
    elif sae_type == "gated":
        # L = ||x - x_hat||^2 + λ ||ReLU(pi_gate)||_1 + ||x - x_hat_frozen||^2
        assert pi is not None, "Gate pre-activations required for gated SAE"
        assert x_hat_aux is not None, "Auxiliary reconstruction required for gated SAE"
        # L1 penalty on ReLU of gate pre-activations
        l1_loss = F.relu(pi).mean()
        # Auxiliary loss: reconstruction from frozen decoder
        # This ensures gradients flow to W_gate and b_gate despite Heaviside step
        aux_loss = (x - x_hat_aux).pow(2).mean()
        total_loss = recon_loss + l1_coef * l1_loss + aux_loss
    
    elif sae_type in ["topk", "batch_topk"]:
        # No sparsity penalty needed (enforced by architecture)
        l1_loss = torch.tensor(0.0)
        aux_loss = torch.tensor(0.0)
        total_loss = recon_loss
    
    # Metrics
    with torch.no_grad():
        fve = 1 - (x - x_hat).var() / x.var()
        l0 = (f.abs() > 1e-6).float().sum(dim=1).mean()
    
    metrics = {
        'loss': total_loss.item(),
        'recon_loss': recon_loss.item(),
        'l1_loss': l1_loss.item() if isinstance(l1_loss, torch.Tensor) else l1_loss,
        'aux_loss': aux_loss.item() if isinstance(aux_loss, torch.Tensor) else 0.0,
        'fve': fve.item(),
        'l0': l0.item(),
    }
    
    return total_loss, metrics